# Physics-Informed Neural Networks: Implementation & Training

This notebook implements the PINN architecture and complete training pipeline for electromagnetic field prediction.

## Setup and Imports

This notebook implements Physics-Informed Neural Networks using the PyTorch framework {cite}`paszke2017automatic_pytorch` for automatic differentiation and GPU acceleration.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import matplotlib.patches as patches
from scipy.special import j0, j1
import warnings
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import seaborn as sns
from mpl_toolkits.axes_grid1 import make_axes_locatable

warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("Physics-Informed Neural Networks Framework Initialized")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

## Recreate Problem Setup

First, we need to instantiate the problem from the previous notebook.

In [ ]:
# Import problem class from previous notebook
class ElectromagneticProblem:
    """2D electromagnetic field problem setup"""

    def __init__(self, domain_size=2.0, wire_position=(0.0, 0.0), current=1.0, mu_0=4*np.pi*1e-7):
        self.domain_size = domain_size
        self.wire_position = wire_position
        self.current = current
        self.mu_0 = mu_0

    def analytical_solution(self, x, y):
        r = np.sqrt((x - self.wire_position[0])**2 + (y - self.wire_position[1])**2)
        r = np.maximum(r, 1e-6)
        B_magnitude = (self.mu_0 * self.current) / (2 * np.pi * r)
        theta = np.arctan2(y - self.wire_position[1], x - self.wire_position[0])
        Hx = -B_magnitude * np.sin(theta) / self.mu_0
        Hy = B_magnitude * np.cos(theta) / self.mu_0
        return Hx, Hy

    def generate_training_data(self, n_data=100):
        x_data = np.random.uniform(-self.domain_size/2, self.domain_size/2, n_data)
        y_data = np.random.uniform(-self.domain_size/2, self.domain_size/2, n_data)
        Hx_data, Hy_data = self.analytical_solution(x_data, y_data)
        return x_data, y_data, Hx_data, Hy_data

    def generate_collocation_points(self, n_domain=1000, n_boundary=100):
        x_domain = np.random.uniform(-self.domain_size/2, self.domain_size/2, n_domain)
        y_domain = np.random.uniform(-self.domain_size/2, self.domain_size/2, n_domain)
        r_from_wire = np.sqrt((x_domain - self.wire_position[0])**2 +
                              (y_domain - self.wire_position[1])**2)
        mask = r_from_wire > 0.1
        x_domain = x_domain[mask]
        y_domain = y_domain[mask]

        boundary_points = []
        x_boundary = np.random.uniform(-self.domain_size/2, self.domain_size/2, n_boundary//2)
        y_boundary_top = np.ones(n_boundary//4) * self.domain_size/2
        y_boundary_bottom = -np.ones(n_boundary//4) * self.domain_size/2
        boundary_points.extend(zip(x_boundary[:n_boundary//4], y_boundary_top))
        boundary_points.extend(zip(x_boundary[n_boundary//4:], y_boundary_bottom))

        y_boundary = np.random.uniform(-self.domain_size/2, self.domain_size/2, n_boundary//2)
        x_boundary_left = -np.ones(n_boundary//4) * self.domain_size/2
        x_boundary_right = np.ones(n_boundary//4) * self.domain_size/2
        boundary_points.extend(zip(x_boundary_left, y_boundary[:n_boundary//4]))
        boundary_points.extend(zip(x_boundary_right, y_boundary[n_boundary//4:]))

        x_boundary, y_boundary = zip(*boundary_points)
        return x_domain, y_domain, np.array(x_boundary), np.array(y_boundary)

# Create problem instance
problem = ElectromagneticProblem(domain_size=2.0, current=1.0)
print("SUCCESS: Problem initialized")

## PINN Architecture for Electromagnetics

### Network Design

A Physics-Informed Neural Network for magnetic field prediction consists of {cite}`goodfellow2016deep`:

1. **Input Layer**: Spatial coordinates $(x, y)$ and material properties $\mu$
2. **Hidden Layers**: Deep neural network with nonlinear activation (tanh)
3. **Output Layer**: Magnetic field components $(H_x, H_y)$
4. **Automatic Differentiation**: Compute $\frac{\partial H_x}{\partial x}$, $\frac{\partial H_y}{\partial y}$, etc. {cite}`paszke2017automatic_pytorch`

### Key Features:

- **Activation Function**: $\tanh$ (smooth derivatives, bounded output)
- **Architecture**: [Input(4) → 32 → 64 → 32 → Output(2)]
- **Derivatives**: Computed via PyTorch autograd for physics loss

In [ ]:
class ElectromagneticPINN(nn.Module):
    """Physics-Informed Neural Network for 2D magnetic field prediction"""

    def __init__(self, layers=[32, 64, 32], activation='tanh'):
        super(ElectromagneticPINN, self).__init__()

        self.activation = activation
        self.layers = layers

        # Build network architecture
        network_layers = []

        # Input layer (x, y, material properties)
        input_dim = 2 + 2  # coordinates + material properties

        # Hidden layers
        prev_dim = input_dim
        for i, layer_dim in enumerate(layers):
            network_layers.append(nn.Linear(prev_dim, layer_dim))
            if i < len(layers) - 1:
                network_layers.append(self._get_activation())
            prev_dim = layer_dim

        # Output layer (Hx, Hy)
        network_layers.append(nn.Linear(prev_dim, 2))

        self.network = nn.Sequential(*network_layers)

    def _get_activation(self):
        """Get activation function"""
        if self.activation == 'tanh':
            return nn.Tanh()
        elif self.activation == 'relu':
            return nn.ReLU()
        elif self.activation == 'sigmoid':
            return nn.Sigmoid()
        else:
            return nn.Tanh()

    def forward(self, x, y, material_props):
        """Forward pass through the neural network"""
        # Combine inputs
        network_input = torch.cat([x, y, material_props], dim=-1)

        # Network forward pass
        output = self.network(network_input)

        # Split into field components
        Hx = output[:, 0:1]  # Magnetic field x-component
        Hy = output[:, 1:2]  # Magnetic field y-component

        return Hx, Hy

    def compute_derivatives(self, x, y, material_props):
        """Compute spatial derivatives for physics loss"""
        x.requires_grad_(True)
        y.requires_grad_(True)

        Hx, Hy = self.forward(x, y, material_props)

        # Compute gradients
        dHx_dx = torch.autograd.grad(Hx, x, grad_outputs=torch.ones_like(Hx),
                                     create_graph=True, retain_graph=True)[0]
        dHx_dy = torch.autograd.grad(Hx, y, grad_outputs=torch.ones_like(Hx),
                                     create_graph=True, retain_graph=True)[0]
        dHy_dx = torch.autograd.grad(Hy, x, grad_outputs=torch.ones_like(Hy),
                                     create_graph=True, retain_graph=True)[0]
        dHy_dy = torch.autograd.grad(Hy, y, grad_outputs=torch.ones_like(Hy),
                                     create_graph=True, retain_graph=True)[0]

        return Hx, Hy, dHx_dx, dHx_dy, dHy_dx, dHy_dy

print("PINN Architecture Defined")
print("SUCCESS: ElectromagneticPINN: Neural network for field prediction")

### Understanding the PINN Architecture

This Physics-Informed Neural Network has special requirements compared to standard neural networks:

#### Input Structure

```python
input_dim = 2 + 2  # coordinates + material properties
```

**Why 4 inputs?**
- **2 spatial coordinates**: (x, y) position in the domain where we evaluate the field
- **2 material properties**: Permeability μ and auxiliary property
  - Allows network to learn material-dependent behavior
  - Essential for multi-material problems (air, iron, copper)

#### Architecture Choice: Tanh Activation

```python
activation='tanh'
```

**Why tanh for PINNs?**
- **Smooth derivatives**: Essential for physics loss (computes ∂H/∂x, ∂H/∂y)
- **Bounded output**: Range [-1, 1] prevents exploding values
- **Symmetric**: Helps with magnetic fields (positive and negative directions)
- ReLU would have problems: non-smooth at 0, unbounded, asymmetric

#### Network Depth

```python
layers=[32, 64, 32]  # 3 hidden layers
```

**Design pattern:**
- **Encoder**: 32 → 64 (expand representation)
- **Decoder**: 64 → 32 (compress to output)
- **Bottleneck**: Wider middle layer captures complex patterns
- Total: ~10K parameters (small enough for fast training)

#### Output Layer Design

```python
network_layers.append(nn.Linear(prev_dim, 2))  # No activation
```

**Why 2 outputs without activation?**
- **Hx, Hy**: Must predict any real-valued magnetic field components
- **No activation**: Regression task (not classification)
- **Separate outputs**: Allows independent x and y components

#### Automatic Differentiation Requirements

```python
x.requires_grad_(True)
y.requires_grad_(True)
```

**Critical for physics loss:**
- Tells PyTorch to track operations on x and y
- Enables computing ∂H/∂x and ∂H/∂y automatically
- Without this, cannot enforce Maxwell's equations
- Creates computational graph for chain rule application

## Physics Loss Computation

The physics loss enforces Maxwell's equations at collocation points throughout the domain.

### Loss Components:

1. **Ampère's Law Loss**:
$$\mathcal{L}_{\text{Ampere}} = \left\| \frac{\partial H_y}{\partial x} - \frac{\partial H_x}{\partial y} - J_z \right\|^2$$

2. **Gauss's Law Loss**:
$$\mathcal{L}_{\text{Gauss}} = \left\| \frac{\partial H_x}{\partial x} + \frac{\partial H_y}{\partial y} \right\|^2$$

3. **Boundary Loss**:
$$\mathcal{L}_{\text{boundary}} = \| H_{\text{pred}} - H_{\text{analytical}} \|^2$$

In [ ]:
class PhysicsLoss:
    """Physics loss computation for Maxwell's equations"""

    def __init__(self, problem):
        self.problem = problem

    def compute_physics_loss(self, model, x_domain, y_domain, x_boundary, y_boundary):
        """Compute physics loss for domain and boundary points"""
        # Convert to tensors
        x_domain_tensor = torch.tensor(x_domain, dtype=torch.float32, requires_grad=True).unsqueeze(1)
        y_domain_tensor = torch.tensor(y_domain, dtype=torch.float32, requires_grad=True).unsqueeze(1)
        x_boundary_tensor = torch.tensor(x_boundary, dtype=torch.float32, requires_grad=True).unsqueeze(1)
        y_boundary_tensor = torch.tensor(y_boundary, dtype=torch.float32, requires_grad=True).unsqueeze(1)

        # Material properties (uniform for this problem)
        mu = torch.ones_like(x_domain_tensor) * self.problem.mu_0
        material_props_domain = torch.cat([mu, torch.ones_like(x_domain_tensor)], dim=1)
        material_props_boundary = torch.cat([mu[:len(x_boundary_tensor)],
                                           torch.ones_like(x_boundary_tensor)], dim=1)

        # Domain physics loss (Maxwell's equations)
        Hx_dom, Hy_dom, dHx_dx, dHx_dy, dHy_dx, dHy_dy = model.compute_derivatives(
            x_domain_tensor, y_domain_tensor, material_props_domain)

        # Current density (point source at wire location)
        r_from_wire = torch.sqrt((x_domain_tensor - self.problem.wire_position[0])**2 +
                                 (y_domain_tensor - self.problem.wire_position[1])**2)
        J_z = self.problem.current * torch.exp(-r_from_wire**2 / 0.01)  # Smoothed point source

        # Ampère's law: ∂Hy/∂x - ∂Hx/∂y = Jz
        ampere_loss = dHy_dx - dHx_dy - J_z

        # Gauss's law for magnetism: ∂Bx/∂x + ∂By/∂y = 0
        # B = μH, so ∂(μHx)/∂x + ∂(μHy)/∂y = 0
        div_B = dHx_dx + dHy_dy  # For uniform μ

        domain_loss = torch.mean(ampere_loss**2) + torch.mean(div_B**2)

        # Boundary loss (far-field conditions)
        Hx_bound, Hy_bound, _, _, _, _ = model.compute_derivatives(
            x_boundary_tensor, y_boundary_tensor, material_props_boundary)

        # Far-field approximation (should match analytical solution at boundaries)
        Hx_analytical, Hy_analytical = self.problem.analytical_solution(x_boundary, y_boundary)
        Hx_analytical_tensor = torch.tensor(Hx_analytical, dtype=torch.float32).unsqueeze(1)
        Hy_analytical_tensor = torch.tensor(Hy_analytical, dtype=torch.float32).unsqueeze(1)

        boundary_loss = torch.mean((Hx_bound - Hx_analytical_tensor)**2) + torch.mean((Hy_bound - Hy_analytical_tensor)**2)

        return domain_loss, boundary_loss

print("SUCCESS: Physics loss computation defined")

### Understanding Physics Loss Computation

The physics loss enforces Maxwell's equations at collocation points. This is the key innovation of PINNs.

#### Automatic Differentiation for PDEs

```python
dHx_dx = torch.autograd.grad(Hx, x, grad_outputs=torch.ones_like(Hx), 
                             create_graph=True, retain_graph=True)[0]
```

**Breaking down the derivative computation:**

- **`torch.autograd.grad(Hx, x, ...)`**: Computes ∂Hx/∂x using chain rule
  - Traces back through network: output → hidden layers → input x
  - Returns gradient of Hx with respect to x coordinate

- **`grad_outputs=torch.ones_like(Hx)`**: Weights for the gradient
  - Ones means: compute gradient for each output independently
  - For multi-output functions, specifies which outputs to differentiate

- **`create_graph=True`**: Build computational graph for gradients
  - Allows computing second derivatives (if needed)
  - Essential for backpropagating through the physics loss

- **`retain_graph=True`**: Keep graph after backward pass
  - Needed because we compute multiple derivatives (dHx/dx, dHx/dy, dHy/dx, dHy/dy)
  - Each gradient computation needs access to the graph

#### Maxwell's Equations as Loss Terms

```python
# Ampère's law: ∂Hy/∂x - ∂Hx/∂y = Jz
ampere_loss = dHy_dx - dHx_dy - J_z

# Gauss's law: ∂Hx/∂x + ∂Hy/∂y = 0
div_B = dHx_dx + dHy_dy
```

**Understanding the physics:**

1. **Ampère's Law** (∇ × H = J):
   - Curl of magnetic field equals current density
   - In 2D: reduces to ∂Hy/∂x - ∂Hx/∂y = Jz
   - `ampere_loss` should be zero if network satisfies physics
   - Non-zero loss → network violates Maxwell's equations

2. **Gauss's Law** (∇ · B = 0):
   - Magnetic field lines form closed loops
   - No magnetic monopoles exist
   - In 2D: ∂Bx/∂x + ∂By/∂y = 0
   - For uniform μ: becomes ∂Hx/∂x + ∂Hy/∂y = 0

#### Current Source Representation

```python
J_z = self.problem.current * torch.exp(-r_from_wire**2 / 0.01)
```

**Why Gaussian smoothing?**
- Real wire: point source (Dirac delta function)
- **Problem**: Point sources have infinite field at center
- **Solution**: Smooth Gaussian distribution
  - `exp(-r²/0.01)`: Decays rapidly away from wire
  - 0.01: Controls smoothness (smaller = sharper, but harder to learn)
- Network can handle smooth functions better than singularities

#### Combined Domain Loss

```python
domain_loss = torch.mean(ampere_loss**2) + torch.mean(div_B**2)
```

**Why square and average?**
- **Squaring**: Makes loss positive, penalizes violations
- **Mean**: Normalizes by number of collocation points
  - Allows comparing losses with different numbers of points
  - Prevents loss from growing with more collocation points
- **Addition**: Both physics laws must be satisfied
  - Could weight them differently (λ₁·Ampère + λ₂·Gauss) if needed

## Training Pipeline

Complete training pipeline with data loss, physics loss, and comprehensive monitoring.

### Training Strategy:

1. **Combined Loss**:
$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{data}} + \lambda(\mathcal{L}_{\text{Ampere}} + \mathcal{L}_{\text{Gauss}} + \mathcal{L}_{\text{boundary}})$$

2. **Optimizer**: Adam {cite}`kingma2014adam` with learning rate 0.01

3. **Hyperparameter**: $\lambda = 1.0$ (equal weight to data and physics)

4. **Epochs**: 300 (optimized for demonstration)

In [ ]:
class PINNTrainer:
    """Training pipeline for Physics-Informed Neural Networks"""

    def __init__(self, model, problem, physics_loss, lambda_physics=1.0):
        self.model = model
        self.problem = problem
        self.physics_loss = physics_loss
        self.lambda_physics = lambda_physics

        # Optimizer
        self.optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

        # Training history
        self.history = {
            'total_loss': [],
            'data_loss': [],
            'physics_loss': [],
            'boundary_loss': []
        }

    def train_epoch(self, x_data, y_data, Hx_data, Hy_data,
                    x_domain, y_domain, x_boundary, y_boundary):
        """Train for one epoch"""
        self.optimizer.zero_grad()

        # Convert data to tensors
        x_data_tensor = torch.tensor(x_data, dtype=torch.float32).unsqueeze(1)
        y_data_tensor = torch.tensor(y_data, dtype=torch.float32).unsqueeze(1)
        Hx_data_tensor = torch.tensor(Hx_data, dtype=torch.float32).unsqueeze(1)
        Hy_data_tensor = torch.tensor(Hy_data, dtype=torch.float32).unsqueeze(1)

        # Material properties for data points
        mu_data = torch.ones_like(x_data_tensor) * self.problem.mu_0
        material_props_data = torch.cat([mu_data, torch.ones_like(x_data_tensor)], dim=1)

        # Data loss
        Hx_pred, Hy_pred = self.model(x_data_tensor, y_data_tensor, material_props_data)
        data_loss = torch.mean((Hx_pred - Hx_data_tensor)**2) + torch.mean((Hy_pred - Hy_data_tensor)**2)

        # Physics loss
        domain_loss, boundary_loss = self.physics_loss.compute_physics_loss(
            self.model, x_domain, y_domain, x_boundary, y_boundary)

        # Total loss
        total_loss = data_loss + self.lambda_physics * (domain_loss + boundary_loss)

        # Backpropagation
        total_loss.backward()
        self.optimizer.step()

        # Store history
        self.history['total_loss'].append(total_loss.item())
        self.history['data_loss'].append(data_loss.item())
        self.history['physics_loss'].append((domain_loss + boundary_loss).item())
        self.history['boundary_loss'].append(boundary_loss.item())

        return total_loss.item(), data_loss.item(), (domain_loss + boundary_loss).item()

    def train(self, epochs=500, print_every=50):
        """Complete training pipeline (optimized for speed)"""
        print(" Starting PINN training...")

        # Generate training data
        x_data, y_data, Hx_data, Hy_data = self.problem.generate_training_data(n_data=30)
        x_domain, y_domain, x_boundary, y_boundary = self.problem.generate_collocation_points()

        print(f"Training data points: {len(x_data)}")
        print(f"Domain collocation points: {len(x_domain)}")
        print(f"Boundary collocation points: {len(x_boundary)}")

        for epoch in range(epochs):
            total_loss, data_loss, physics_loss = self.train_epoch(
                x_data, y_data, Hx_data, Hy_data,
                x_domain, y_domain, x_boundary, y_boundary
            )

            if epoch % print_every == 0:
                print(f"Epoch {epoch:4d}: Total Loss = {total_loss:.6f}, "
                      f"Data Loss = {data_loss:.6f}, Physics Loss = {physics_loss:.6f}")

        print("SUCCESS: Training completed!")
        return self.history

print("SUCCESS: PINN training pipeline defined")

## Model Training

Let's train the PINN on our electromagnetic problem with optimized parameters for faster execution.

In [ ]:
# Create and train the model (optimized for speed)
model = ElectromagneticPINN(layers=[32, 64, 32], activation='tanh')
physics_loss_computer = PhysicsLoss(problem)
trainer = PINNTrainer(model, problem, physics_loss_computer, lambda_physics=1.0)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Train the model (reduced epochs for faster execution)
history = trainer.train(epochs=300, print_every=50)

## Training Progress Visualization

Let's visualize the training progress to understand how the PINN learns.

In [ ]:
def plot_training_history(history):
    """Plot training loss curves"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle('PINN Training Progress', fontsize=16, fontweight='bold')

    # Total loss
    axes[0, 0].plot(history['total_loss'], 'b-', linewidth=2, label='Total Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Total Loss')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()

    # Data loss
    axes[0, 1].plot(history['data_loss'], 'r-', linewidth=2, label='Data Loss')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].set_title('Data Loss')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()

    # Physics loss
    axes[1, 0].plot(history['physics_loss'], 'g-', linewidth=2, label='Physics Loss')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Loss')
    axes[1, 0].set_title('Physics Loss')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()

    # Combined view
    axes[1, 1].semilogy(history['total_loss'], 'b-', linewidth=2, label='Total')
    axes[1, 1].semilogy(history['data_loss'], 'r-', linewidth=2, label='Data')
    axes[1, 1].semilogy(history['physics_loss'], 'g-', linewidth=2, label='Physics')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Loss (log scale)')
    axes[1, 1].set_title('All Losses (Log Scale)')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend()

    plt.tight_layout()
    plt.show()

    plot_training_history(history)

## Summary

In this notebook, we implemented:

1. **PINN Architecture**:
- Neural network with 3 hidden layers [32, 64, 32]
- Automatic differentiation for physics loss
- Total of ~10K parameters

2. **Physics Loss**:
- Ampère's law enforcement
- Gauss's law for magnetism
- Boundary condition matching

3. **Training Pipeline**:
- Combined data + physics loss
- Adam optimizer with lr=0.01
- Training for 300 epochs
- Loss monitoring and visualization

**Key Observations**:
- Both data loss and physics loss decrease during training
- Physics loss enforces Maxwell's equations throughout domain
- Only 30 labeled points required (vs thousands for traditional DL)

In the next notebook (06d), we will evaluate the trained PINN and visualize the predicted magnetic field distributions.